# Wildfire Burn Severity Mapping with Spectral Indices

Companion notebook for the **EcoGeo Tutor** tutorial: *Python: Wildfire Burn Severity Mapping with Spectral Indices*.

Implements a complete burned-area workflow using Sentinel-2 imagery: NBR, dNBR,
USGS severity classification, and export.

**Data you'll need:** pre- and post-fire Sentinel-2 L2A bands (B08, B12, SCL) —
free from [Copernicus Open Access Hub](https://scihub.copernicus.eu) or Google Earth Engine.
Update the file paths below to match your own downloads.


## Setup

In [ ]:
!pip install numpy rasterio matplotlib geopandas -q


## 1. Load Sentinel-2 bands

B8 = NIR (10m), B12 = SWIR2 (20m, resample to 10m). We need pre- and post-fire scenes.

In [ ]:
import numpy as np
import rasterio
import rasterio.plot
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def load_band(path):
    with rasterio.open(path) as src:
        return src.read(1).astype("float32"), src.profile, src.crs

nir_pre,  profile, crs = load_band("pre_fire_B08.tif")
swir_pre, *_           = load_band("pre_fire_B12_10m.tif")
nir_post, *_           = load_band("post_fire_B08.tif")
swir_post, *_          = load_band("post_fire_B12_10m.tif")


## 2. Mask clouds using the SCL band

SCL values: 4=vegetation, 5=bare soil, 6=water, 8-10=clouds/shadows. Only keep pixels clear in BOTH dates.

In [ ]:
def valid_mask(scl_path, valid_classes=(4, 5)):
    scl, *_ = load_band(scl_path)
    return np.isin(scl, valid_classes)

valid_pre  = valid_mask("pre_fire_SCL.tif")
valid_post = valid_mask("post_fire_SCL.tif")
valid      = valid_pre & valid_post


## 3. Compute NBR

NBR = (NIR - SWIR) / (NIR + SWIR). Healthy vegetation is high; burned areas are low.

In [ ]:
def nbr(nir, swir, mask=None):
    with np.errstate(divide="ignore", invalid="ignore"):
        result = (nir - swir) / (nir + swir)
    result = np.where(nir + swir == 0, 0, result)
    if mask is not None:
        result[~mask] = np.nan
    return result

nbr_pre  = nbr(nir_pre,  swir_pre,  valid)
nbr_post = nbr(nir_post, swir_post, valid)

print(f"Pre-fire  NBR range: {np.nanmin(nbr_pre):.3f} - {np.nanmax(nbr_pre):.3f}")
print(f"Post-fire NBR range: {np.nanmin(nbr_post):.3f} - {np.nanmax(nbr_post):.3f}")


## 4. Compute dNBR and classify severity

dNBR = NBR_pre minus NBR_post. Positive values indicate burned area; USGS thresholds (Key & Benson 2006) classify severity.

In [ ]:
dnbr = nbr_pre - nbr_post

def classify_severity(dnbr):
    classes = np.zeros_like(dnbr, dtype="uint8")
    classes[dnbr < -0.25]                      = 1  # enhanced regrowth
    classes[(dnbr >= -0.25) & (dnbr < 0.10)]   = 2  # unburned
    classes[(dnbr >= 0.10)  & (dnbr < 0.27)]   = 3  # low severity
    classes[(dnbr >= 0.27)  & (dnbr < 0.44)]   = 4  # moderate-low
    classes[(dnbr >= 0.44)  & (dnbr < 0.66)]   = 5  # moderate-high
    classes[dnbr >= 0.66]                       = 6  # high severity
    classes[np.isnan(dnbr)]                     = 0  # masked
    return classes

severity = classify_severity(dnbr)

burned_area_km2 = np.sum(severity >= 3) * (10 * 10) / 1e6
print(f"Total burned area: {burned_area_km2:.1f} km2")


## 5. Visualize

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

im1 = axes[0].imshow(nbr_pre,  cmap="RdYlGn", vmin=-1, vmax=1)
axes[0].set_title("NBR - Pre-fire", fontweight="bold")
axes[0].axis("off")
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

im2 = axes[1].imshow(nbr_post, cmap="RdYlGn", vmin=-1, vmax=1)
axes[1].set_title("NBR - Post-fire", fontweight="bold")
axes[1].axis("off")
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

cmap_sev = mcolors.ListedColormap(
    ["white", "#15803D", "#A3E635", "#FDE047", "#F97316", "#DC2626", "#7F1D1D"])
bounds = [0, 1, 2, 3, 4, 5, 6, 7]
norm   = mcolors.BoundaryNorm(bounds, cmap_sev.N)
im3 = axes[2].imshow(severity, cmap=cmap_sev, norm=norm)
axes[2].set_title("Burn Severity (dNBR)", fontweight="bold")
axes[2].axis("off")
cbar = plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04, ticks=[0.5,1.5,2.5,3.5,4.5,5.5,6.5])
cbar.ax.set_yticklabels(["Masked","Regrowth","Unburned","Low","Mod-Low","Mod-High","High"], fontsize=8)

plt.suptitle("Wildfire Burn Severity - Sentinel-2 dNBR Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("wildfire_severity_map.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Export

In [ ]:
out_profile = profile.copy()
out_profile.update(dtype=rasterio.uint8, count=1, nodata=0)

with rasterio.open("burn_severity.tif", "w", **out_profile) as dst:
    dst.write(severity, 1)

print("Saved: burn_severity.tif")


## Limitations

- **Dark surface confusion** — shadows, deep water, and dark soil can resemble burned areas in NBR. Validate against known reference areas.
- **Cloud contamination** — even small cloud fractions corrupt results; use SCL masking rigorously.
- **No temporal persistence** — a single post-fire image doesn't capture recovery; time-series dNBR tracks recovery over months and years.
- **Smoke during active fires** — optical sensors are blocked by smoke; integrate Sentinel-1 SAR for near-real-time monitoring.

---
*Companion notebook for the EcoGeo Tutor tutorial on rcafe.vercel.app*
